# Subset selection for pushT demos

Pick a subset of the 78 episodes in `datasets/pushT/sharded-demo` by clicking the initial-frame thumbnail of each episode, then materialize the subset into a new ShardedHDF5-compatible directory `datasets/pushT/sharded-subset-<x>/` (you choose the suffix).

**Cells:**
1. Load metadata + frame-0 from every shard.
2. Render the click-to-toggle thumbnail grid.
3. Set the subset name and write the new dataset.

In [1]:
from pathlib import Path
import json
import h5py
import numpy as np

SRC_DIR = Path('/home/mim-server/datasets/pushT/sharded-demo')
DST_ROOT = Path('/home/mim-server/datasets/pushT')

with open(SRC_DIR / 'metadata.json') as f:
    src_meta = json.load(f)
src_meta

{'num_shards': 78,
 'total_episodes': 78,
 'image_shape': [256, 256, 3],
 'action_shape': [3],
 'relative_actions': True,
 'delta_direction': 'future',
 'dt_source': 'timestamp'}

In [2]:
# Enumerate (shard_idx, ep_idx) and load frame 0 of each episode.
num_shards = src_meta['num_shards']
episodes = []  # list of dicts: {'shard': int, 'ep': int, 'length': int, 'frame0': np.ndarray}

for s in range(num_shards):
    p = SRC_DIR / f'shard_{s:04d}.h5'
    with h5py.File(p, 'r') as f:
        try:
            lengths = f['episode_lengths'][:]
        except Exception:
            lengths = np.array([f['episode_lengths'][()]])
        for ep_idx, ep_len in enumerate(lengths):
            frame0 = f['images'][ep_idx, 0]  # (H, W, 3) uint8
            episodes.append({
                'shard': s,
                'ep': int(ep_idx),
                'length': int(ep_len),
                'frame0': np.array(frame0),
            })

print(f'Loaded {len(episodes)} episodes from {num_shards} shards.')
print(f'Frame shape: {episodes[0]["frame0"].shape}, dtype: {episodes[0]["frame0"].dtype}')

Loaded 78 episodes from 78 shards.
Frame shape: (256, 256, 3), dtype: uint8


## Click-to-toggle thumbnail grid

Each tile shows `shard:ep · T=length` and the initial frame. Click to toggle inclusion (green border = selected). The live count + indices update below.

In [3]:
import io
import base64
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

THUMB_SIZE = 128       # px, displayed size
COLS = 8               # tiles per row

selected = set()       # set of indices into `episodes`

def _png_data_uri(arr, size=THUMB_SIZE):
    img = Image.fromarray(arr).resize((size, size), Image.BILINEAR)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

def _tile_html(idx, on):
    e = episodes[idx]
    border = '4px solid #2ecc71' if on else '4px solid #ddd'
    bg = '#eafaf1' if on else '#fff'
    return (
        f'<div style="border:{border};background:{bg};padding:2px;'
        f'border-radius:6px;text-align:center;font-family:monospace;font-size:11px;">'
        f'<img src="{_png_data_uri(e["frame0"])}" width="{THUMB_SIZE}" height="{THUMB_SIZE}" />'
        f'<div>s{e["shard"]:02d}:e{e["ep"]} · T={e["length"]}</div>'
        f'<div>idx={idx}</div>'
        f'</div>'
    )

buttons = []
for idx in range(len(episodes)):
    btn = widgets.Button(
        description='',
        layout=widgets.Layout(width=f'{THUMB_SIZE+16}px', height=f'{THUMB_SIZE+50}px', padding='0'),
        tooltip=f'Toggle episode {idx}',
    )
    btn._idx = idx
    buttons.append(btn)

# Each button gets an HTML preview as its child via a paired widget.
tile_htmls = [widgets.HTML(_tile_html(i, False)) for i in range(len(episodes))]

status = widgets.HTML('<b>Selected: 0</b>')
selected_out = widgets.Output()

def _refresh_status():
    status.value = f'<b>Selected: {len(selected)} / {len(episodes)}</b>'
    with selected_out:
        selected_out.clear_output()
        print('selected indices:', sorted(selected))

def _on_click(idx):
    def handler(_btn):
        if idx in selected:
            selected.remove(idx)
        else:
            selected.add(idx)
        tile_htmls[idx].value = _tile_html(idx, idx in selected)
        _refresh_status()
    return handler

# Stack each tile = HTML preview + transparent button underneath
tile_boxes = []
for i, (btn, html) in enumerate(zip(buttons, tile_htmls)):
    btn.on_click(_on_click(i))
    # Click target is the small button label; full-area click works via a Box overlay.
    # Simplest reliable variant: just put a labeled button under the HTML.
    btn.description = 'toggle'
    tile_boxes.append(widgets.VBox([html, btn], layout=widgets.Layout(margin='2px')))

# Lay out as a grid of rows.
rows = []
for r in range(0, len(tile_boxes), COLS):
    rows.append(widgets.HBox(tile_boxes[r:r+COLS]))

def _select_all(_):
    selected.clear()
    selected.update(range(len(episodes)))
    for i in range(len(episodes)):
        tile_htmls[i].value = _tile_html(i, True)
    _refresh_status()

def _clear_all(_):
    selected.clear()
    for i in range(len(episodes)):
        tile_htmls[i].value = _tile_html(i, False)
    _refresh_status()

btn_all = widgets.Button(description='Select all')
btn_none = widgets.Button(description='Clear')
btn_all.on_click(_select_all)
btn_none.on_click(_clear_all)
controls = widgets.HBox([btn_all, btn_none, status])

_refresh_status()
display(controls, widgets.VBox(rows), selected_out)

Output()

## Materialize the subset

Set `SUBSET_TAG` to a short descriptor that identifies how the subset was chosen (e.g. `'top-left-init'`, `'short-eps'`, `'manual-pick-2026-05-17'`). The new dataset is written to `/home/mim-server/datasets/pushT/sharded-subset-<SUBSET_TAG>/`.

Each output shard contains exactly the selected episodes from one source shard; shard files are renumbered contiguously from `shard_0000.h5` onwards so `ShardedHDF5Dataset` can iterate them with no gaps.

In [6]:
SUBSET_TAG = 'center-sparse'        # <-- edit me
OVERWRITE  = False                # set True to allow clobbering an existing subset dir

selected_episodes = sorted(
    (episodes[i]['shard'], episodes[i]['ep']) for i in selected
)
assert selected_episodes, 'Nothing selected — go back and click some tiles.'

dst_dir = DST_ROOT / f'sharded-subset-{SUBSET_TAG}'
if dst_dir.exists():
    if not OVERWRITE:
        raise FileExistsError(f'{dst_dir} exists. Set OVERWRITE=True or pick a new SUBSET_TAG.')
    import shutil
    shutil.rmtree(dst_dir)
dst_dir.mkdir(parents=True, exist_ok=False)

# Group selected eps by source shard so each output shard sources from exactly one input shard.
from collections import defaultdict
by_shard = defaultdict(list)
for s, e in selected_episodes:
    by_shard[s].append(e)

def _copy_subset_shard(src_path, dst_path, ep_idxs):
    """Read src shard, write a new shard containing only `ep_idxs`. Preserves schema.

    All top-level datasets whose first dim equals num_episodes_src are sliced;
    `episode_lengths` is re-emitted from the sliced lengths; other root datasets
    are copied through unchanged. Attrs are preserved except `num_episodes`.
    """
    ep_idxs = list(ep_idxs)
    with h5py.File(src_path, 'r') as fi, h5py.File(dst_path, 'w') as fo:
        num_src = int(fi.attrs.get('num_episodes', fi['episode_lengths'].shape[0] if 'episode_lengths' in fi else 1))
        for k, v in fi.attrs.items():
            if k != 'num_episodes':
                fo.attrs[k] = v
        fo.attrs['num_episodes'] = len(ep_idxs)

        for name in fi.keys():
            ds = fi[name]
            if name == 'episode_lengths':
                try:
                    src_lengths = ds[:]
                except Exception:
                    src_lengths = np.array([ds[()]])
                kept = np.asarray([int(src_lengths[i]) for i in ep_idxs], dtype=np.int32)
                fo.create_dataset('episode_lengths', data=kept)
                continue

            # Per-episode datasets have first dim = num_src.
            if ds.ndim >= 1 and ds.shape[0] == num_src:
                sub = ds[ep_idxs, ...]
                # Trim the time dim to max kept episode length when sensible.
                if 'episode_lengths' in fi and ds.ndim >= 2:
                    try:
                        src_lengths = fi['episode_lengths'][:]
                    except Exception:
                        src_lengths = np.array([fi['episode_lengths'][()]])
                    kept_max = int(max(int(src_lengths[i]) for i in ep_idxs))
                    if sub.shape[1] > kept_max:
                        sub = sub[:, :kept_max]
                # Preserve chunking shape where possible; let h5py pick otherwise.
                chunks = ds.chunks
                if chunks is not None:
                    chunks = (min(chunks[0], sub.shape[0]),) + tuple(
                        min(c, s) for c, s in zip(chunks[1:], sub.shape[1:])
                    )
                fo.create_dataset(
                    name, data=sub, dtype=ds.dtype,
                    chunks=chunks if chunks else None,
                    compression=ds.compression, compression_opts=ds.compression_opts,
                )
            else:
                # Non-per-episode dataset — copy through.
                fo.create_dataset(name, data=ds[()])

out_idx = 0
manifest = []  # (out_shard_idx, src_shard, [ep_idxs])
for src_shard in sorted(by_shard.keys()):
    ep_idxs = sorted(by_shard[src_shard])
    src_path = SRC_DIR / f'shard_{src_shard:04d}.h5'
    dst_path = dst_dir / f'shard_{out_idx:04d}.h5'
    _copy_subset_shard(src_path, dst_path, ep_idxs)
    manifest.append({'out_shard': out_idx, 'src_shard': src_shard, 'src_ep_idxs': ep_idxs})
    out_idx += 1

new_meta = dict(src_meta)
new_meta['num_shards'] = out_idx
new_meta['total_episodes'] = sum(len(m['src_ep_idxs']) for m in manifest)
new_meta['subset_of'] = str(SRC_DIR)
new_meta['subset_tag'] = SUBSET_TAG
new_meta['subset_manifest'] = manifest

with open(dst_dir / 'metadata.json', 'w') as f:
    json.dump(new_meta, f, indent=2)

print(f'Wrote {out_idx} shards covering {new_meta["total_episodes"]} episodes to {dst_dir}')

Wrote 27 shards covering 27 episodes to /home/mim-server/datasets/pushT/sharded-subset-center-sparse


## Verify with `ShardedHDF5Dataset`

Quick smoke check — loads the new subset directory through the same dataset class the trainer uses.

In [7]:
import sys
sys.path.insert(0, str(Path.cwd().parent))
from dreamerv4uwm.datasets import ShardedHDF5Dataset

ds = ShardedHDF5Dataset(
    data_dir=str(dst_dir),
    window_size=16,
    stride=1,
    split='train',
    train_fraction=1.0,
    shuffle_windows=False,
)
sample = ds[0]
print('image:', sample['image'].shape, sample['image'].dtype, 'range', float(sample['image'].min()), float(sample['image'].max()))
print('action:', sample['action'].shape, sample['action'].dtype)
print('episodes loaded:', len(ds.episode_lengths))
print('total windows:', len(ds))

Train split: 2995 windows from 27 episodes
image: torch.Size([16, 3, 256, 256]) torch.float32 range 0.0 0.8509804010391235
action: torch.Size([16, 3]) torch.float32
episodes loaded: 27
total windows: 2995
